In [1]:
from copy import deepcopy
import torch
import sys
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda import amp
from spikingjelly.activation_based import functional, surrogate, neuron, layer
from spikingjelly.activation_based.model import parametric_lif_net
from spikingjelly.datasets.dvs128_gesture import DVS128Gesture
from torch.utils.data import DataLoader
import time
import os
import argparse
import datetime

In [2]:
torch.manual_seed(1)

In [3]:
T = 16
b = 8
j = 8
lr = 0.001
epochs = 20
channels = 128

data_dir = os.path.expanduser('~/datasets/DVSGesture/')

In [4]:
device = 'cuda:0'

In [5]:
class DVSGestureNet(nn.Module):
    def __init__(self, channels=128, spiking_neuron: callable=None, is_seperable=False, kernel_size=3, **kwargs):
        super().__init__()

        conv = []
        ## First Layer
        conv.append(layer.Conv2d(2, channels, kernel_size=kernel_size, 
                                         padding=1, bias=False))
        conv.append(layer.BatchNorm2d(channels))
        conv.append(layer.MaxPool2d(2, 2))

        ## Middle Layers
        for i in range(4):
            if is_seperable:
                conv.append(layer.Conv2d(channels, channels,
                                         kernel_size=kernel_size, groups=channels,
                                         padding=1, bias=False)
                )
                conv.append(layer.BatchNorm2d(channels))
                conv.append(layer.Conv2d(channels, channels, kernel_size=1))
            else:
                conv.append(layer.Conv2d(channels, channels, kernel_size=kernel_size, 
                                         padding=1, bias=False))
                conv.append(layer.BatchNorm2d(channels))
                
            conv.append(spiking_neuron(**deepcopy(kwargs)))
            conv.append(layer.MaxPool2d(2, 2))


        self.conv = nn.Sequential(
            *conv, 
            layer.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = layer.Linear(in_features=channels, out_features=11)
        
    def forward(self, x: torch.Tensor):
        x = self.conv(x).mean(0).squeeze()
        x = self.fc(x)
            
        return x


In [6]:
train_set = DVS128Gesture(root=data_dir, train=True, data_type='frame', frames_number=T, split_by='number')
test_set = DVS128Gesture(root=data_dir, train=False, data_type='frame', frames_number=T, split_by='number')

The directory [/home/tahaf/datasets/DVSGesture/frames_number_16_split_by_number] already exists.
The directory [/home/tahaf/datasets/DVSGesture/frames_number_16_split_by_number] already exists.


In [7]:
train_data_loader = torch.utils.data.DataLoader(
    dataset=train_set,
    batch_size=b,
    shuffle=True,
    drop_last=True,
    num_workers=j,
    pin_memory=True
)

test_data_loader = torch.utils.data.DataLoader(
    dataset=test_set,
    batch_size=b,
    shuffle=True,
    drop_last=False,
    num_workers=j,
    pin_memory=True
)

In [8]:
scaler = amp.GradScaler()

In [9]:
def check_model(kernel_size, is_seperable):
    print(f'Results for kernel size = {kernel_size} and seperable convolution = {is_seperable}')
    max_test_acc = -1

    net = DVSGestureNet(
        channels=channels,
        kernel_size=kernel_size,
        spiking_neuron=neuron.LIFNode,
        surrogate_function=surrogate.ATan(),
        is_seperable=is_seperable,
        detach_reset=True
    )
    net.to(device)
    num_params = sum(p.numel() for p in net.parameters())
    functional.set_step_mode(net, step_mode='m')

    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    
    start_time = time.time()
    for epoch in range(epochs):
        net.train()
        train_loss = 0
        train_acc = 0
        train_samples = 0
        for frame, label in train_data_loader:
            optimizer.zero_grad()
            frame = frame.to(device)
            frame = frame.transpose(0, 1)  # [N, T, C, H, W] -> [T, N, C, H, W]
            label = label.to(device)
            label_onehot = F.one_hot(label, 11).float()
    
            if scaler is not None:
                with amp.autocast():
                    out_fr = net(frame)
                    loss = F.cross_entropy(out_fr, label_onehot)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                out_fr = net(frame)
                loss = F.cross_entropy(out_fr, label_onehot)
                loss.backward()
                optimizer.step()
    
            train_samples += label.numel()
            train_loss += loss.item() * label.numel()
            train_acc += (out_fr.argmax(1) == label).float().sum().item()
    
            functional.reset_net(net)
    
        train_loss /= train_samples
        train_acc /= train_samples
    
        lr_scheduler.step()
    
        net.eval()
        test_loss = 0
        test_acc = 0
        test_samples = 0
        with torch.no_grad():
            for frame, label in test_data_loader:
                frame = frame.to(device)
                frame = frame.transpose(0, 1)  # [N, T, C, H, W] -> [T, N, C, H, W]
                label = label.to(device)
                label_onehot = F.one_hot(label, 11).float()
                out_fr = net(frame)
                loss = F.cross_entropy(out_fr, label_onehot)
                test_samples += label.numel()
                test_loss += loss.item() * label.numel()
                test_acc += (out_fr.argmax(1) == label).float().sum().item()
                functional.reset_net(net)
        test_loss /= test_samples
        test_acc /= test_samples
        max_test_acc = max(max_test_acc, test_acc)
        
        print(f'epoch = {epoch}, train_loss ={train_loss: .4f}, train_acc ={train_acc: .4f}, test_loss ={test_loss: .4f}, test_acc ={test_acc: .4f}, max_test_acc ={max_test_acc: .4f}')
    end_time = time.time()
    total_time = end_time - start_time
    print(f'total time: {total_time}s')
    print('-'*1000)

    return {
        'accuracy': max_test_acc,
        'total_time': total_time,
        'num_params': num_params,
    }

In [10]:
kernel_sizes = [3, 5, 7]
results = {}

In [ ]:
for kernel_size in kernel_sizes:
    for is_seperable in [False, True]:
        result = check_model(kernel_size=kernel_size, is_seperable=is_seperable)
        results[kernel_size][is_seperable] = result

Results for kernel size = 3 and seperable convolution = False
